In [70]:
#!pip install folium

In [1]:
import requests
import json
import pandas as pd
import folium
import os

# URL Overpass (miroir stable)
OVERPASS_URL = "https://overpass.kumi.systems/api/interpreter"


# 🔹 1. Récupération des données depuis OSM
def fetch_osm_data(city, filename):
    query = f"""
    [out:json][timeout:25];
    area["name"="{city}"]->.searchArea;

    (
      node["amenity"~"school|restaurant|hospital|pharmacy|atm"](area.searchArea);
      way["amenity"~"school|restaurant|hospital|pharmacy|atm"](area.searchArea);

      node["tourism"="hotel"](area.searchArea);
      way["tourism"="hotel"](area.searchArea);
    );

    out center;
    """

    response = requests.get(OVERPASS_URL, params={'data': query}, timeout=60)

    if response.status_code == 200:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"✅ Données sauvegardées pour {city} dans {filename}")
    else:
        raise ValueError(f"❌ Erreur Overpass: {response.status_code}")


# 🔹 2. Nettoyage et transformation en DataFrame
def clean_dataframe(city, filename):
    with open(filename, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            raise ValueError(f"❌ Le fichier {filename} est vide.")
        try:
            data = json.loads(content)
        except json.JSONDecodeError:
            raise ValueError(f"❌ JSON invalide dans {filename}")

    elements = data.get("elements", [])
    rows = []

    for el in elements:
        tags = el.get("tags", {})
        lat = el.get("lat") or el.get("center", {}).get("lat")
        
        lon = el.get("lon") or el.get("center", {}).get("lon")

        category = tags.get("amenity") or tags.get("tourism")
        name = tags.get("name") or tags.get("operator") or tags.get("brand")

        # ✅ Correction : ne pas exclure si name est vide
        if lat and lon and category:
            rows.append({
                "city": city,
                "lat": lat,
                "lon": lon,
                "category": category,
                "name": name if name else f"{category}_unnamed",
                "source": "OSM"
            })

    df = pd.DataFrame(rows)

    # 🔹 Nettoyage
    df = df.drop_duplicates(subset=["city", "lat", "lon", "category", "name"])
    df = df.reset_index(drop=True)

    return df


# 🔹 3. Sauvegarde
def save_dataframe(df, filename_csv="../data/pos_cotonou.csv", filename_excel="../data/pos_cotonou.xlsx"):
    os.makedirs("../data", exist_ok=True)

    df.to_csv(filename_csv, index=False, encoding="utf-8")
    df.to_excel(filename_excel, index=False)

    print(f"✅ Données sauvegardées :")
    print(f"   - CSV : {filename_csv}")
    print(f"   - Excel : {filename_excel}")


# 🔹 4. Visualisation sur carte (multi-catégories)
def show_on_map(df, city):
    df_filtered = df[df["city"] == city]

    if df_filtered.empty:
        print(f"❌ Aucune donnée pour {city}")
        return None

    lat0, lon0 = df_filtered.iloc[0]["lat"], df_filtered.iloc[0]["lon"]
    m = folium.Map(location=[lat0, lon0], zoom_start=13)

    colors = {
        "school": "blue",
        "restaurant": "red",
        "hospital": "green",
        "hotel": "purple",
        "pharmacy": "orange",
        "atm": "black"
    }

    for _, row in df_filtered.iterrows():
        folium.Marker(
            location=[row["lat"], row["lon"]],
            popup=f"{row['name']} ({row['category']})",
            tooltip=row["name"],
            icon=folium.Icon(color=colors.get(row["category"], "gray"))
        ).add_to(m)

    map_file = f"../data/{city}_map.html"
    m.save(map_file)

    print(f"🗺️ Carte sauvegardée dans {map_file}")
    return m


# 🚀 5. Exécution principale
if __name__ == "__main__":
    city = "Cotonou"

    json_file = f"../data/osm_{city}.json"

    # Étape 1 : récupération
    fetch_osm_data(city, json_file)

    # Étape 2 : nettoyage
    df = clean_dataframe(city, json_file)

    # Vérification rapide
    print("\n📊 Aperçu des données :")
    print(df.head())

    print("\n📊 Répartition des catégories :")
    print(df["category"].value_counts())

    # Étape 3 : sauvegarde
    save_dataframe(df)

    # Étape 4 : carte
    show_on_map(df, city)

✅ Données sauvegardées pour Cotonou dans ../data/osm_Cotonou.json

📊 Aperçu des données :
      city       lat       lon    category               name source
0  Cotonou  6.347745  2.403413       hotel      Arcada Marina    OSM
1  Cotonou  6.349609  2.386550         atm            Ecobank    OSM
2  Cotonou  6.356871  2.404173       hotel     Chant d'oiseau    OSM
3  Cotonou  6.358407  2.431501    pharmacy  Pharmacie Jonquet    OSM
4  Cotonou  6.359506  2.419645  restaurant         Costa Rica    OSM

📊 Répartition des catégories :
category
restaurant        192
school            153
hotel              90
pharmacy           81
hospital           23
atm                14
driving_school     10
music_school        1
Name: count, dtype: int64
✅ Données sauvegardées :
   - CSV : ../data/pos_cotonou.csv
   - Excel : ../data/pos_cotonou.xlsx
🗺️ Carte sauvegardée dans ../data/Cotonou_map.html
